In [2]:
import random
import copy

COLORS = ['Red', 'Blue', 'Green', 'Yellow'] 
VALUES = [str(i) for i in range(10)] + ['Skip'] 

class Card: 
    def __init__(self, color, value):
        self.color = color
        self.value = value

    def __repr__(self):
        return f"{self.color} {self.value}"

#  Deck Generation 
def generate_deck():
    deck = [Card(color, value) for color in COLORS for value in VALUES]
    random.shuffle(deck)
    return deck

def get_valid_moves(hand, top_card):
    valid_moves = []
    for card in hand:
        
        # Rule 1: A player may play a card if it matches the color OR number 
        if card.color == top_card.color or card.value == top_card.value:
            valid_moves.append(card)
            
    return valid_moves

# State Transition 
def apply_move(state, player_key, move):
    new_state = copy.deepcopy(state)
    
    # Rule 2: If no valid card exists, player must draw 1 card 
    if move == "Draw":
        if len(new_state['deck']) > 0:
            drawn_card = new_state['deck'].pop()
            new_state[player_key].append(drawn_card)
    else:
        for i, card in enumerate(new_state[player_key]):
            if card.color == move.color and card.value == move.value:
                new_state[player_key].pop(i)
                break
        new_state['top_card'] = move
        
    return new_state

def simulate_random_game(initial_state):
    c_st = copy.deepcopy(initial_state)
    p = ['Player1', 'Player2', 'Ai'] 
    t_idx = 0
    
    while True:
        c_p = p[t_idx % 3]
        hnd = c_st[c_p]
        
        # Rule 4: player wins when cards = 0 
        if len(hnd) == 0:
            print(f"\ngame over! {c_p} wins!")
            break 
            
        print(f"\ntop card: {c_st['top_card']}\n")
        print(f"{c_p} hand:")
        for c in hnd:
            print(c)
            
        print(f"\n{c_p} decision(all possible decisions considered at depth 1):\n")
        
        if c_p == 'Player1':
            _, c_m = mm(c_st, 3, 1, c_p, t_idx)
        elif c_p == 'Player2':
            _, c_m = em(c_st, 3, c_p, t_idx)
        else:
            _, c_m = mm(c_st, 3, 1, c_p, t_idx)
            
        if not c_m:
            c_m = "Draw"
            
        c_st = apply_move(c_st, c_p, c_m)
        
        # Rule 3: if player plays skip, next player's turn is skipped 
        if c_m != "Draw" and c_m.value == 'Skip':
            t_idx += 2 
        else:
            t_idx += 1 
            
        if len(c_st['deck']) == 0:
            print("\ndraw deck is empty. it's a tie!")
            break

def evaluate_state(st, p, strat="baseline"):
    ops = [x for x in ['Player1', 'Player2', 'Ai'] if x != p]
    
    my_c = len(st[p])
    op_c = (len(st[ops[0]]) + len(st[ops[1]])) / 2
    
    skp = sum(1 for c in st[p] if c.value == 'Skip')
    
    w_my, w_op, w_sk = 5, 2, 3
    
    if strat == "offensive":
        w_my, w_op, w_sk = 7, 2, 2
    elif strat == "defensive":
        w_my, w_op, w_sk = 4, 4, 5
        
    return 50 - (w_my * my_c) + (w_op * op_c) + (w_sk * skp)
    
def mm(st, dp, is_mx, p_k, t_idx):
    if dp == 0 or not st['Player1'] or not st['Player2'] or not st['Ai']:
        return evaluate_state(st, p_k, "defensive"), 0
        
    p = ['Player1', 'Player2', 'Ai']
    c_p = p[t_idx % 3]
    v_m = get_valid_moves(st[c_p], st['top_card'])
    
    if not v_m:
        v_m = ["Draw"]
        
    b_m = 0
    
    if is_mx:
        
        m_v = -float('inf')
        
        for m in v_m:
            n_st = apply_move(st, c_p, m)
            n_t = t_idx + 2 if m != "Draw" and m.value == 'Skip' else t_idx + 1
            v, _ = mm(n_st, dp - 1, 0, p_k, n_t)
            
            if dp == 3:
                print(f"play: {m}")
                print(f"expected score: {round(v, 1)}\n")
                
            if v > m_v:
                m_v = v
                b_m = m
        return m_v, b_m
    else:
        m_v = float('inf')
        for m in v_m:
            n_st = apply_move(st, c_p, m)
            n_t = t_idx + 2 if m != "Draw" and m.value == 'Skip' else t_idx + 1
            n_p = p[n_t % 3]
            n_mx = 1 if n_p == p_k else 0
            v, _ = mm(n_st, dp - 1, n_mx, p_k, n_t)
            if v < m_v:
                m_v = v
                b_m = m
        return m_v, b_m
        
def em(st, dp, p_k, t_idx):
    if dp == 0 or not st['Player1'] or not st['Player2'] or not st['Ai']:
        return evaluate_state(st, p_k, "offensive"), 0
        
    p = ['Player1', 'Player2', 'Ai']
    c_p = p[t_idx % 3]
    v_m = get_valid_moves(st[c_p], st['top_card'])
    
    if not v_m:
        v_m = ["Draw"]
        
    b_m = 0
    
    if c_p == p_k:
        m_v = -float('inf')
        
        for m in v_m:
            if m == "Draw":
                e_v = 0
                d_c = len(st['deck'])
                if d_c > 0:
                    pr = 1.0 / d_c
                    for i in range(d_c):
                        n_st = copy.deepcopy(st)
                        d_o = n_st['deck'].pop(i)
                        n_st[c_p].append(d_o)
                        v, _ = em(n_st, dp - 1, p_k, t_idx + 1)
                        e_v += pr * v
                        
                    if dp == 3:
                        print(f"play: draw")
                        print(f"expected score: {round(e_v, 1)}\n")
                        
                    if e_v > m_v:
                        m_v = e_v
                        b_m = m
                else:
                    v = evaluate_state(st, p_k, "offensive")
                    
                    if dp == 3:
                        print(f"play: draw")
                        print(f"expected score: {round(v, 1)}\n")
                        
                    if v > m_v:
                        m_v = v
                        b_m = m
            else:
                n_st = apply_move(st, c_p, m)
                n_t = t_idx + 2 if m.value == 'Skip' else t_idx + 1
                v, _ = em(n_st, dp - 1, p_k, n_t)
                
                if dp == 3:
                    print(f"play: {m}")
                    print(f"expected score: {round(v, 1)}\n")
                    
                if v > m_v:
                    m_v = v
                    b_m = m
        return m_v, b_m
    else:
        e_v = 0
        l_m = len(v_m)
        for m in v_m:
            n_st = apply_move(st, c_p, m)
            n_t = t_idx + 2 if m != "Draw" and m.value == 'Skip' else t_idx + 1
            v, _ = em(n_st, dp - 1, p_k, n_t)
            e_v += v
        return (e_v / l_m) if l_m > 0 else 0, 0   
 # testing if game works           
game_deck = generate_deck()

test_state = {
    'Player1': [game_deck.pop() for _ in range(5)],
    'Player2': [game_deck.pop() for _ in range(5)],
    'Ai': [game_deck.pop() for _ in range(5)],
    'top_card': game_deck.pop(),
    'deck': game_deck
}

simulate_random_game(test_state)


top card: Yellow 1

Player1 hand:
Yellow 9
Red 1
Blue 7
Blue 3
Blue 1

Player1 decision(all possible decisions considered at depth 1):

play: Yellow 9
expected score: 50.0

play: Red 1
expected score: 50.0

play: Blue 1
expected score: 50.0


top card: Yellow 9

Player2 hand:
Blue 2
Red 9
Yellow 6
Green 1
Red 4

Player2 decision(all possible decisions considered at depth 1):

play: Red 9
expected score: 29.0

play: Yellow 6
expected score: 26.1


top card: Red 9

Ai hand:
Yellow Skip
Blue Skip
Red 6
Red 0
Green 2

Ai decision(all possible decisions considered at depth 1):

play: Red 6
expected score: 56.0

play: Red 0
expected score: 56.0


top card: Red 6

Player1 hand:
Red 1
Blue 7
Blue 3
Blue 1

Player1 decision(all possible decisions considered at depth 1):

play: Red 1
expected score: 50.0


top card: Red 1

Player2 hand:
Blue 2
Yellow 6
Green 1
Red 4

Player2 decision(all possible decisions considered at depth 1):

play: Green 1
expected score: 36.0

play: Red 4
expected score: 

the function calculates a score using this baseline formula :

### score = 50 - w_my(c_ai) + w_op(c_opp) + w_sk(s)

#### the variables:

c_ai: cards in the ai's hand. this is subtracted because having fewer cards brings you closer to winning.

c_opp: average cards held by the two opponents. this is added because it is good when opponents have a lot of cards.

s: skip cards held. this is added because skip cards give you turn control.

50: a constant value to keep the overall score positive.


#### the strategy weights:

baseline (w_my=5, w_op=2, w_sk=3): standard, balanced play.

offensive (w_my=7, w_op=2, w_sk=2): heavily penalizes holding your own cards to force aggressive card shedding.

defensive (w_my=4, w_op=4, w_sk=5): strongly rewards hoarding skip cards and keeping opponent card counts high.